# AED — Dataset Unificado ICSAP-UTI, São Paulo 2022-2024

Challenge FIAP-Oracle 2026 · Projeto Clariti

Análise Exploratória de Dados do `dataset_unificado_sp_2022_2024.parquet`: internações com UTI em São Paulo, classificadas por ICSAP (Internações por Condições Sensíveis à Atenção Primária).

Este notebook segue o Guia_Statistical_Methods (passos 9-16): a AED aqui é a base que alimenta a Preparação Estatística (passo 9) mais adiante — distribuição, métricas, missing e outliers já saem estruturados pra isso.

## 0. Regras da casa (decididas antes de qualquer análise)

Essas regras vêm de decisões já tomadas e documentadas em `Documentacao_Dataset_ICSAP_UTI_SP.docx` (seções 5 e 9). Repetidas aqui porque toda análise deste notebook depende delas:

1. **O dataset não é 1 linha = 1 internação.** É 1 linha = 1 bucket agregado (município + diagnóstico + ano + mês). A contagem real de internações está em `qtd_internacoes_uti`. **Sempre somar essa coluna, nunca contar linha.**
2. **Taxa sempre por 10 mil habitantes**, nunca contagem bruta, pra poder comparar município grande com município pequeno.
3. **Mediana e IQR**, não média e desvio-padrão — dado de custo e permanência hospitalar tem cauda longa (poucos casos muito caros/longos puxam a média pra cima).
4. **Outlier clínico não se remove cego.** Uma internação de 300 dias pode ser exatamente o caso caro que interessa pro gestor, não erro de dado.
5. **Unidade de análise depende da pergunta**: município x ano pra comparar município (evita ruído de número pequeno em cidade pequena), mês x estado inteiro pra sazonalidade (evita o mesmo problema, só que no eixo temporal — município x mês individual tem 50% dos casos com 5 internações ou menos, é ruído demais pra comparar cidade por cidade).

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

df = pd.read_parquet("dados/dataset_unificado_sp_2022_2024.parquet")
print(f"{len(df):,} linhas, {df.shape[1]} colunas")
df.head()

463,299 linhas, 20 colunas


,municipio_6,diag_princ,qtd_internacoes_uti,soma_val_tot,soma_dias_perm,soma_uti_mes_to,ano,mes,uf_residencia,reside_fora_sp,qtd_estabelecimentos,qtd_leitos_existentes,qtd_leitos_sus,qtd_leitos_contratados,qtd_leitos_nao_sus,populacao,nome_municipio,csap,grupo_csap_cod,grupo_csap_nome
0,110003,I850,1,4011.20,6,3,2022,1,RO,True,NaN,NaN,NaN,NaN,NaN,NaN,None,não,None,None
1,110011,R100,1,2211.98,7,3,2022,1,RO,True,NaN,NaN,NaN,NaN,NaN,NaN,None,não,None,None
2,110092,D359,1,7750.23,5,2,2022,1,RO,True,NaN,NaN,NaN,NaN,NaN,NaN,None,não,None,None
3,120045,C910,1,12972.41,32,22,2022,1,AC,True,NaN,NaN,NaN,NaN,NaN,NaN,None,não,None,None
4,120060,Q211,1,14270.28,9,5,2022,1,AC,True,NaN,NaN,NaN,NaN,NaN,NaN,None,não,None,None


## 1. Perfil geral do dataset

Antes de qualquer métrica, o básico: quantas linhas, que período, que granularidade, quanto do estado o dado cobre. Isso vira o "raio-x" que todo o resto da AED assume como verdade.

In [2]:
print("Período:", df["ano"].min(), "a", df["ano"].max())
print("Meses por ano:", sorted(int(m) for m in df["mes"].unique()))
print()
print("Combinações (município, diag_princ, ano, mês) = linhas do dataset:", f"{len(df):,}")
print("Códigos de diagnóstico distintos (diag_princ):", df["diag_princ"].nunique())
print("Códigos de município distintos (municipio_6):", df["municipio_6"].nunique())

Período: 2022 a 2024
Meses por ano: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Combinações (município, diag_princ, ano, mês) = linhas do dataset: 463,299
Códigos de diagnóstico distintos (diag_princ): 6242
Códigos de município distintos (municipio_6): 2175


In [3]:
total_internacoes = df["qtd_internacoes_uti"].sum()
print(f"Total de internações com UTI (soma de qtd_internacoes_uti, não linha): {total_internacoes:,}")
print()
print("Por ano:")
print(df.groupby("ano")["qtd_internacoes_uti"].sum().apply(lambda x: f"{x:,}"))

Total de internações com UTI (soma de qtd_internacoes_uti, não linha): 813,422

Por ano:
ano
2022    251,743
2023    271,679
2024    290,000
Name: qtd_internacoes_uti, dtype: object


In [4]:
residentes_sp = df.loc[~df["reside_fora_sp"], "qtd_internacoes_uti"].sum()
residentes_fora = df.loc[df["reside_fora_sp"], "qtd_internacoes_uti"].sum()

municipios_sp = df.loc[~df["reside_fora_sp"], "municipio_6"].nunique()
municipios_fora = df.loc[df["reside_fora_sp"], "municipio_6"].nunique()

print(f"Municípios de SP que aparecem no dataset: {municipios_sp} (de 645 no total do estado)")
print(f"Municípios de fora de SP que aparecem: {municipios_fora}")
print()
print(f"Internações de residente de SP: {residentes_sp:,} ({residentes_sp/total_internacoes:.1%})")
print(f"Internações de residente de fora de SP: {residentes_fora:,} ({residentes_fora/total_internacoes:.1%})")

Municípios de SP que aparecem no dataset: 645 (de 645 no total do estado)
Municípios de fora de SP que aparecem: 1530

Internações de residente de SP: 805,118 (99.0%)
Internações de residente de fora de SP: 8,304 (1.0%)


In [5]:
icsap = df.loc[df["csap"] == "sim", "qtd_internacoes_uti"].sum()
nao_icsap = df.loc[df["csap"] == "não", "qtd_internacoes_uti"].sum()

print(f"ICSAP: {icsap:,} ({icsap/total_internacoes:.1%})")
print(f"Não-ICSAP: {nao_icsap:,} ({nao_icsap/total_internacoes:.1%})")
print()
print("% ICSAP por ano:")
tab = df.groupby(["ano", "csap"])["qtd_internacoes_uti"].sum().unstack(fill_value=0)
tab["%_icsap"] = (tab["sim"] / (tab["sim"] + tab["não"]) * 100).round(1)
print(tab)

ICSAP: 152,069 (18.7%)
Não-ICSAP: 661,353 (81.3%)

% ICSAP por ano:


csap     não    sim  %_icsap
ano                         
2022  205763  45980     18.3
2023  219019  52660     19.4
2024  236571  53429     18.4


### Leitura da etapa 1

- O dataset cobre os 645 municípios de SP mais gente de fora do estado internada aqui (SP é polo de atendimento de alta complexidade — decisão de não filtrar isso, seção 5.3 do doc).
- 813.422 internações com UTI no total do período, crescendo ano a ano (251.743 → 271.679 → 290.000). Não dá pra saber ainda se é crescimento real de demanda ou melhoria de cobertura/registro do SIH — fica como pergunta em aberto pra próxima etapa.
- 18,7% das internações são ICSAP, estável entre 18,3% e 19,4% nos 3 anos — não tem uma tendência clara de piora ou melhora no período, o que por si é uma informação (destaca a ausência de mudança).
- Próxima etapa (2): quantificar os nulos formalmente, e explicar a causa de cada um (já sabemos 3 causas diferentes: fora de SP, bug de 9 linhas, gap CNES-LT — mas isso ainda não foi visualizado nem tabulado dentro do notebook).

## 2. Missing (valores ausentes)

Antes de calcular qualquer taxa ou média, precisa saber onde o dado falta — senão uma conta com nulo silencioso vira número errado sem ninguém perceber.

**O que "missing" não é aqui**: não é "esse dado não existe no mundo real". É "essa linha do dataset não tem um valor preenchido nessa coluna". As causas podem ser bem diferentes entre si, e tratar todo nulo como a mesma coisa esconde informação. Por isso a etapa não para em "quanto % é nulo" — precisa decompor **por quê**.

In [6]:
nulos = df.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
pct = (nulos / len(df) * 100).round(2)

resumo_nulos = pd.DataFrame({"linhas_nulas": nulos, "pct_do_total": pct})
resumo_nulos

,linhas_nulas,pct_do_total
grupo_csap_cod,384762,83.05
grupo_csap_nome,384762,83.05
qtd_leitos_nao_sus,49737,10.74
qtd_leitos_existentes,49737,10.74
qtd_leitos_sus,49737,10.74
qtd_leitos_contratados,49737,10.74
qtd_estabelecimentos,7833,1.69
nome_municipio,7833,1.69
populacao,7833,1.69


A tabela tem 3 famílias, não 2 — a maior de todas (83%) é `grupo_csap_cod`/`grupo_csap_nome`, e ela merece um parágrafo à parte porque **não é o mesmo tipo de problema** das outras duas.

### 2.0 A família de 83% não é "dado faltando" — é nulo por design

`grupo_csap_cod` só é preenchido quando `csap = "sim"` (a internação é ICSAP). Quando não é, o grupo de causa não existe, então o valor é nulo — isso já estava definido assim desde a classificação (seção 5.5 do `.docx`, e no Dicionário de Dados, seção 10). Não tem nada "faltando" aqui: é o mesmo tipo de nulo que aparece quando uma pergunta de formulário é "se sim, qual?" e a resposta foi não.

Essa distinção tem nome em estatística: **missing por design** (ou MNAR estrutural) é diferente de **missing data problem**. O segundo é um defeito — um dado que deveria existir e não existe, por falha de coleta/registro. O primeiro é esperado pela própria definição da variável, e "tratar" isso (preencher com algo) destruiria informação real, não recuperaria nada. As duas famílias que sobram (7.833 e 49.737) SÃO do segundo tipo — por isso recebem atenção daqui pra frente, e essa de 83% não.

In [7]:
# confere a regra: grupo_csap_cod é nulo se e só se csap == "não"
inconsistente = ((df["csap"] == "não") != df["grupo_csap_cod"].isna()).sum()
print(f"Linhas onde a regra 'grupo_csap_cod nulo <=> csap=não' NÃO vale: {inconsistente}")

Linhas onde a regra 'grupo_csap_cod nulo <=> csap=não' NÃO vale: 0


Restam duas famílias de nulo que SÃO problema de dado real (não por design): uma com 7.833 linhas (1,7%), outra com 49.737 linhas (10,7%).

### 2.1 Decompondo o grupo de 7.833 (`populacao`, `nome_municipio`, `qtd_estabelecimentos`)

In [8]:
pop_nula = df["populacao"].isna()

fora_sp = (pop_nula & df["reside_fora_sp"]).sum()
bug_sp = (pop_nula & ~df["reside_fora_sp"]).sum()

print(f"Total nulo: {pop_nula.sum():,}")
print(f"  Residente de fora de SP (esperado — não faz sentido ter população/CNES de SP pra quem não mora aqui): {fora_sp:,}")
print(f"  Residente de SP sem população (bug de ordem de merge, documentado no .docx seção 2.8, não corrigido): {bug_sp:,}")

Total nulo: 7,833
  Residente de fora de SP (esperado — não faz sentido ter população/CNES de SP pra quem não mora aqui): 7,824
  Residente de SP sem população (bug de ordem de merge, documentado no .docx seção 2.8, não corrigido): 9


Dos 7.833, 7.824 são exatamente o esperado — não tem população de São Paulo pra registrar pra quem mora em outro estado. Os 9 restantes são um bug real de implementação (ordem dos merges em `Montar_Dataset_Unificado.py`), pequeno o bastante (0,002% do dataset) pra não valer o retrabalho de corrigir agora, mas registrado.

`qtd_estabelecimentos` tem nulo nas exatas mesmas linhas de `populacao` (conferido, não assumido) — faz sentido, ambos dependem da mesma junção com a dimensão de município.

### 2.2 Decompondo o grupo de 49.737 (`qtd_leitos_existentes` e as outras 3 colunas de leito)

In [9]:
leito_nulo = df["qtd_leitos_existentes"].isna()

junto_com_grupo_781 = (leito_nulo & df["qtd_estabelecimentos"].isna()).sum()
so_leito = (leito_nulo & df["qtd_estabelecimentos"].notna()).sum()

print(f"Total nulo: {leito_nulo.sum():,}")
print(f"  Mesmas linhas do grupo anterior (fora de SP + bug): {junto_com_grupo_781:,}")
print(f"  SÓ leito nulo, estabelecimento preenchido (gap CNES-LT vs CNES-ST): {so_leito:,}")

Total nulo: 49,737
  Mesmas linhas do grupo anterior (fora de SP + bug): 7,833
  SÓ leito nulo, estabelecimento preenchido (gap CNES-LT vs CNES-ST): 41,904


Aqui está o achado que vale registrar: 41.904 linhas (9,0% do dataset) têm estabelecimento de saúde cadastrado (CNES-ST) mas nenhum registro de leito (CNES-LT) naquele mês. Não é "0 leitos" — é ausência de registro. Já investigamos isso a fundo (ver `.docx` seção 9): pode ser porque o estabelecimento realmente não tem leito (ex: posto de saúde básico), ou porque o gestor local não atualizou o cadastro de leitos naquele mês. Os dois casos geram exatamente o mesmo sintoma no dado, e não dá pra distinguir um do outro só com o que temos.

Até aqui, decompusemos o nulo por **causa** (fora de SP, bug, gap de cadastro). O capítulo 7 do livro-texto pede mais que isso antes de decidir o que fazer: pede pra classificar o **mecanismo** (MCAR/MAR/MNAR), o **padrão** (univariada/monotônica/arbitrária), a **intenção** (intencional/acidental) e a **visibilidade** do nulo. É isso que vem a seguir, testado contra os dados reais, não assumido.

### 2.3 Padrão de ausência: univariada, monotônica ou arbitrária? (segundo você mesmo, pág. 313)

O livro (cap. 7, pág. 313) separa o padrão de ausência em três tipos: **univariada** (só uma coluna tem nulo, o resto está completo), **monotônica** (dá pra ordenar as colunas de um jeito que, se a coluna J tem nulo numa linha, todas as colunas depois dela nessa ordem também têm — um "degrau"), e **arbitrária** (nulo espalhado sem essa estrutura, qualquer combinação de coluna pode faltar).

Aqui não é univariada (já são 3 famílias diferentes de coluna com nulo). A hipótese a testar: será que toda linha com `qtd_estabelecimentos` nulo também tem `qtd_leitos_existentes` nulo? Se sim, é monotônica (o nulo de leito "engloba" o nulo de estabelecimento). Se não, é arbitrária.

In [10]:
so_estabelecimento_nulo_leito_preenchido = (
    df["qtd_estabelecimentos"].isna() & df["qtd_leitos_existentes"].notna()
).sum()
print(f"Linhas com estabelecimento nulo MAS leito preenchido: {so_estabelecimento_nulo_leito_preenchido}")
print("(esperado 0 se o padrão for monotônico — nulo de leito 'engloba' o de estabelecimento)")

Linhas com estabelecimento nulo MAS leito preenchido: 0
(esperado 0 se o padrão for monotônico — nulo de leito 'engloba' o de estabelecimento)


Deu 0. **Tradução**: "não existe nenhuma linha onde eu sei o número de leitos mas não sei se existe estabelecimento" — a informação sempre falta primeiro no nível mais "de cima" (estabelecimento) antes de faltar no nível mais "de baixo" (leito). Isso é padrão **monotônico** (em degrau): as 7.833 linhas sem estabelecimento estão *dentro* das 49.737 linhas sem leito (conferido na seção 2.2), e tem mais 41.904 linhas que só faltam no leito. Não é coincidência de rótulo — reflete a hierarquia real dos dois cadastros: CNES-ST (estabelecimento) é o registro "pai", CNES-LT (leito) é o "filho" (só existe leito se o estabelecimento existir no cadastro), então faz sentido que o nulo do filho seja, no mínimo, tão grande quanto o do pai.

### 2.4 Mecanismo: MCAR, MAR ou MNAR? (segundo você mesmo, pág. 311)

O livro (pág. 311) diz que essa classificação importa antes de decidir o que fazer com o nulo. Rápido, pra não confundir os três:

- **MCAR** (Missing Completely At Random): o motivo do nulo não tem NENHUMA relação com nenhuma variável, nem observada nem não observada. É "sorte", tipo perder uma amostra de sangue no laboratório por acidente.
- **MAR** (Missing At Random): o motivo do nulo TEM relação com alguma variável que a gente observa no próprio dataset. Não é sorte, é regra condicional: "dado que eu já sei X, dá pra prever o nulo de Y".
- **MNAR** (Missing Not At Random): o motivo do nulo depende do PRÓPRIO valor que está faltando (ex: gente com renda muito alta não responde "qual sua renda", o nulo depende do dado que não temos).

A hipótese natural (e a que foi levantada primeiro) seria MCAR: "faltou, deve ser aleatório". Vamos testar isso contra os dados, não assumir.

In [11]:
tabela_cruzada = pd.crosstab(df["reside_fora_sp"], df["populacao"].isna())
tabela_cruzada.columns = ["populacao_preenchida", "populacao_nula"]
tabela_cruzada.index.name = "reside_fora_sp"
tabela_cruzada

,populacao_preenchida,populacao_nula
reside_fora_sp,,
False,455466,9
True,0,7824


**Contraste com o que parecia ser**: se fosse MCAR, o nulo de `populacao` apareceria espalhado meio aleatoriamente entre `reside_fora_sp=True` e `reside_fora_sp=False`, sem padrão. Não é isso que a tabela mostra: toda linha com `reside_fora_sp=True` tem população nula (7.824 de 7.824), e toda linha com `reside_fora_sp=False` tem população preenchida, exceto 9 (os do bug já documentado). Zero exceção fora dessas 9.

**O pulo do gato**: a pergunta certa pra distinguir MCAR de MAR não é "tem padrão?", é "dá pra PREVER o nulo usando uma variável que eu já tenho no dataset?". Aqui dá, com quase 100% de acerto, usando só `reside_fora_sp`. Isso descarta MCAR por definição (MCAR exige que não dê pra prever usando NADA, nem observado nem não observado). É **MAR**, e de um jeito bem forte (determinístico, não só correlacionado).

**Tradução do código**: "agrupa as linhas em 2x2, mora fora de SP ou não (linha) contra população nula ou não (coluna), e conta quantas caem em cada combinação". Os 4 números da tabela são a prova da dependência.

Na prática esse MAR é quase um caso-limite: como o motivo é 100% determinístico e conhecido (é decisão de escopo do projeto, não falha aleatória de registro), ele se comporta mais perto do `grupo_csap_cod` da seção 2.0 (nulo por design) do que de um MAR "clássico" de pesquisa (ex: perda de acompanhamento em estudo longitudinal). A diferença prática: aqui dá pra explicar o nulo com 100% de certeza, não só com probabilidade alta.

In [12]:
sub = df[df["qtd_estabelecimentos"].notna()].copy()
sub["leito_nulo"] = sub["qtd_leitos_existentes"].isna()
sub["porte"] = pd.qcut(
    sub["qtd_estabelecimentos"], q=4,
    labels=["Q1 (menor)", "Q2", "Q3", "Q4 (maior)"], duplicates="drop"
)

taxa_nulo_por_porte = (sub.groupby("porte", observed=True)["leito_nulo"].mean() * 100).round(1)
taxa_nulo_por_porte

porte
Q1 (menor)    36.3
Q2             0.2
Q3             0.0
Q4 (maior)     0.0
Name: leito_nulo, dtype: float64

Mesmo teste pro gap de leito (41.904 linhas), agora usando o porte do município (aproximado pelo número de estabelecimentos de saúde, dividido em quartis) em vez de `reside_fora_sp`.

**Contraste**: se fosse MCAR, a taxa de nulo seria parecida nos 4 quartis. Não é: o quartil dos municípios menores (Q1) tem 36,3% de nulo, contra praticamente 0% nos quartis 3 e 4 (municípios maiores). Descarta MCAR de novo, pelo mesmo motivo da família anterior: dá pra prever (bem menos que 100%, mas de forma clara) usando uma variável observada.

Aqui a classificação é mais delicada que a da população. É **MAR** em relação a `qtd_estabelecimentos`/porte, isso está provado pelos números acima. Mas não dá pra descartar **MNAR**: se um motivo real for "município pequeno tem cadastro de leito pior justamente porque tem MENOS leito de verdade pra cadastrar" (ou seja, o motivo do nulo se liga ao próprio número que falta), isso seria MNAR, e não tem como provar nem descartar isso só com os dados que temos (precisaria de uma fonte externa que confirme o número real de leito desses municípios). Fica documentado como incerteza real, não resolvida artificialmente.

### 2.5 Intenção: intencional ou acidental? (segundo você mesmo, pág. 314)

O livro (pág. 314) separa o nulo por intenção: **intencional** (decisão consciente de projeto/coleta gerou o nulo) ou **acidental** (falha não planejada).

Aqui as duas famílias reais caem em lados opostos:

- `populacao`/`nome_municipio`/`qtd_estabelecimentos` (fora de SP): **intencional**. O projeto delimitou o escopo geográfico em São Paulo desde o desenho (ver `.docx` seção 5), e a decisão documentada em `Montar_Dataset_Unificado.py` foi manter essas linhas no dataset (não filtrar), só sem dado de contexto de município, porque esse dado nunca fez parte do escopo de coleta.
- `qtd_leitos_*` (gap CNES-LT): **acidental**. Ninguém decidiu deixar de cadastrar leito de município pequeno — é falha/atraso de atualização de um cadastro público mantido de forma descentralizada, não uma escolha do projeto.

### 2.6 Visibilidade e propagação (segundo você mesmo, pág. 314)

O livro (pág. 314) também separa **visibilidade** (dá pra perceber que falta dado só olhando?) em explícito (o nulo aparece, tipo célula vazia/`NaN`) ou implícito (o nulo está disfarçado de valor válido, e passa despercebido se não for procurado ativamente). E fala de **propagação**: nulo causado por limite prático de amostragem que reduz a granularidade do dado, com dois sub-tipos — informação **censurada** (o valor real existe, mas foi resumido numa faixa, ex.: idade registrada como "65+") e informação **truncada** (a observação inteira foi removida da amostra, por acidente ou de propósito).

Aplicando aos nossos dois casos:

- **Visibilidade**: explícito nos dois. O nulo aparece como `NaN` de verdade no parquet (não tem valor sentinela disfarçado tipo "0" ou "-1" fingindo ser dado válido), dá pra ver com `.isna()` direto, sem precisar investigar escondido.
- **Propagação**: não se aplica a nenhum dos dois casos, e vale registrar o porquê, não só dizer "não é". Não é censura: `populacao` e `qtd_leitos_existentes` são valores numéricos exatos que faltam por completo, não valores reais resumidos numa faixa. Não é truncamento: nenhuma linha foi removida da amostra por causa disso, e essa é uma decisão que já está documentada no código-fonte antes mesmo dessa análise. O cabeçalho do `Montar_Dataset_Unificado.py` registra a decisão (Vitor, 18/08) de **não filtrar** residentes de fora de SP, exatamente pra evitar truncamento silencioso: "(...) Essas linhas ficam sem populacao/CNES de contexto (...), mas continuam no dataset, e ganham 2 colunas pra ficarem identificaveis (...). Assim quem for analisar decide na hora se inclui ou exclui, em vez de eu decidir escondido no ETL." Ou seja, o truncamento foi uma opção considerada e descartada de propósito, antes mesmo desse notebook existir.

### 2.7 O que a literatura acadêmica diz (pesquisa complementar)

O livro-texto é a base, mas vale cruzar com literatura de referência pra não ficar preso a uma única fonte:

- A classificação de padrão (univariada/monotônica/arbitrária) usada aqui bate com a definição formal de van Buuren, *Flexible Imputation of Missing Data* (referência padrão da área de imputação): padrão monotônico é exatamente "se a variável J está ausente numa linha, todas as variáveis depois dela nessa ordem também estão" — o mesmo "degrau" testado e confirmado na seção 2.3.
- Sobre o que fazer com nulo MAR especificamente em registro administrativo de saúde, o guia da AHRQ (*Managing Missing Data in Patient Registries*, NCBI Bookshelf) é direto: pra ANÁLISE INFERENCIAL (regressão, teste, modelo), deixar como "caso completo" (ignorar as linhas com nulo) não é recomendado quando o nulo é MAR — o indicado seria imputação múltipla, MLE, ou ponderação por probabilidade inversa, dependendo do caso.

Isso é importante e honesto de registrar: a decisão de "manter nulo e documentar" (próxima seção) não é o tratamento formal recomendado pra QUALQUER uso do dado — ela é adequada especificamente porque, nesta AED, `qtd_leitos_*` não entra em nenhuma taxa nem indicador calculado (é só perfil descritivo do município). Se um passo mais à frente (Preparação Estatística/ML, passos 9 a 16) usar leito como variável de entrada de modelo ou teste, "manter nulo" deixa de ser suficiente, e a escolha ali precisa ser imputação múltipla ou método equivalente, não uma imputação simples nem um descarte ingênuo de linha.

**Fontes**: van Buuren, S. *Flexible Imputation of Missing Data*, seção 4.1 — https://stefvanbuuren.name/fimd/missing-data-pattern.html. AHRQ / NCBI Bookshelf, *Managing Missing Data in Patient Registries* — https://www.ncbi.nlm.nih.gov/books/NBK493610/.

### 2.8 Decisão de tratamento por coluna

| Coluna | Mecanismo | Padrão | Intenção | Visibilidade / Propagação | % do dataset | Decisão |
|---|---|---|---|---|---|---|
| `populacao`, `nome_municipio`, `qtd_estabelecimentos` | MAR (determinístico via `reside_fora_sp`) | Monotônico (subconjunto do nulo de leito) | Intencional (escopo do projeto) | Explícito; não é propagação | 1,69% | Abaixo de 5% (regra pág. 313) → mantém nulo, documentado. Não corrige (fora de escopo) nem imputa (não dá pra inventar população de outro estado). |
| `qtd_leitos_existentes`/`sus`/`contratados`/`nao_sus` | MAR quanto ao porte do município (MNAR não descartado) | Monotônico (engloba a família anterior + gap próprio) | Acidental (gap de cadastro) | Explícito; não é propagação | 10,74% | Acima de 5% — a regra do livro NÃO cobre "pode ignorar". Mantém nulo por enquanto, com motivo documentado abaixo, não "ignora sem pensar". |

Pro `qtd_leitos_*`, passamos pela lista de técnicas do livro (pág. ~317) uma por uma antes de decidir:

- **Imputação constante/arbitrária** (0 ou mediana geral): descartada. Preencher com 0 finge "sem leito nenhum" pra município que provavelmente tem leito só não cadastrado; preencher com a mediana geral aplica o perfil do município médio/grande exatamente no grupo onde o nulo está concentrado (Q1, menor porte).
- **Imputação simples por média do próprio município**: descartada. 41.904 linhas é volume grande demais pra assumir que "os outros meses estavam certos", e um problema de cadastro tende a se repetir, não ser pontual.
- **Imputação estratificada por porte**: reduziria o erro mas não resolve. Dentro do próprio Q1 (onde a taxa de nulo é 36,3%) ainda tem variação real de leito entre município, e é justamente ali que mais precisaria acertar.
- **Regressão / MICE**: descartada por enquanto. Pede assumir que estabelecimento/população/ano preveem leito de forma confiável, o que não foi testado, e seria arriscado num cadastro que já mostrou ter gap de registro.
- **Remoção** (linha ou coluna): descartada. Jogaria fora dado real de internação (`qtd_internacoes_uti`, `csap`) só pra resolver o nulo de uma coluna secundária que nem é o indicador central da tese.

Decisão final: **mantém nulo, documentado, sem imputação nesta etapa da AED**, porque `qtd_leitos_*` não entra em nenhuma taxa/indicador calculado aqui (seção 2.7). Se for usado como variável de entrada de modelo ou teste nos passos 9 a 16, a decisão volta a ser aberta ali, e nesse caso a literatura recomenda imputação múltipla ou método equivalente, não imputação simples nem "manter nulo".

Nenhuma coluna teve valor substituído/imputado nessa etapa — a decisão foi entender e documentar a causa e o mecanismo, não maquiar o nulo.

### Leitura da etapa 2

- 3 famílias de nulo, e são coisas bem diferentes entre si: `grupo_csap_cod` (83%) é nulo por design (não é problema), `populacao`/`estabelecimento` (1,7%) é MAR intencional e determinístico (escopo de projeto), `qtd_leitos_*` (10,7%) é MAR/possível MNAR acidental (gap de cadastro).
- Testar o mecanismo (não assumir) mudou a conclusão: a hipótese inicial de MCAR não se sustentou em nenhuma das duas famílias reais — as duas têm nulo que dá pra prever com uma variável observada (`reside_fora_sp` ou porte do município), o que por definição já descarta MCAR.
- A regra dos 5% do livro cobre só a família de população (1,69%). A de leito (10,74%) passa do limite e exigiu decisão de tratamento de verdade, não só "ignorar" — a decisão final foi manter nulo e documentar, com a ressalva de que isso vale só enquanto leito não entra em nenhuma conta/indicador desta AED.
- Próxima etapa (3): unidade de análise município x ano, com as métricas definidas na etapa 0 — ainda falta decidir como anualizar `qtd_leitos_*`/`qtd_estabelecimentos` (que são mensais) pro nível de ano, dado que uma parte relevante do dado é nula.

## 3. Distribuição das métricas centrais (unidade: município x ano)

Essa etapa monta a tabela município x ano (a unidade de comparação geográfica decidida na etapa 0, regra 5), com as métricas centrais da tese, e olha como elas se distribuem entre os 645 municípios de SP.

Duas decisões tomadas antes de escrever qualquer código (documentadas também no `.docx`, seção 9):

- **Exclui residente de fora de SP** dessa tabela: essas linhas não têm população nem CNES (nulo intencional, seção 2.1), não dá pra posicionar num ranking de município de SP nem calcular taxa. Continuam no dataset principal, só não entram nessa tabela específica.
- **Sempre soma antes de dividir** (nunca média de média): `qtd_internacoes_uti`, `soma_val_tot` e `soma_uti_mes_to` são somados no nível município x ano antes de qualquer taxa ser calculada, seguindo a regra 0.1/0.2 da seção 0. É exatamente o conceito de **média ponderada** que o livro trata no capítulo 3 (Estimativas de localização: média, média ponderada, mediana, moda...).

In [13]:
df_sp_mun = df[~df["reside_fora_sp"]].copy()

metricas_centrais = (
    df_sp_mun.groupby(["municipio_6", "ano"])
    .agg(
        n_internacoes=("qtd_internacoes_uti", "sum"),
        soma_val_tot=("soma_val_tot", "sum"),
        soma_uti_mes_to=("soma_uti_mes_to", "sum"),
        populacao=("populacao", "median"),
        nome_municipio=("nome_municipio", "first"),
    )
    .reset_index()
)

n_icsap = (
    df_sp_mun[df_sp_mun["csap"] == "sim"]
    .groupby(["municipio_6", "ano"])["qtd_internacoes_uti"]
    .sum()
    .rename("n_icsap")
)

metricas_centrais = metricas_centrais.merge(n_icsap, on=["municipio_6", "ano"], how="left")
metricas_centrais["n_icsap"] = metricas_centrais["n_icsap"].fillna(0)

print(f"{len(metricas_centrais):,} combinações município x ano (máximo possível: 645 x 3 = 1.935)")
metricas_centrais.head()

1,935 combinações município x ano (máximo possível: 645 x 3 = 1.935)


,municipio_6,ano,n_internacoes,soma_val_tot,soma_uti_mes_to,populacao,nome_municipio,n_icsap
0,350010,2022,357,2409622.85,2436,34687.0,Adamantina - SP,81.0
1,350010,2023,340,2034558.55,1740,34687.0,Adamantina - SP,61.0
2,350010,2024,335,2116637.02,1873,35642.0,Adamantina - SP,59.0
3,350020,2022,64,624747.47,381,4351.0,Adolfo - SP,10.0
4,350020,2023,53,431688.78,242,4351.0,Adolfo - SP,20.0


`n_icsap` usa `fillna(0)` de propósito, e **não é o mesmo tipo de decisão do leito**: se um município-ano não aparece no filtro `csap == "sim"`, é porque teve exatamente zero internação ICSAP naquele ano (dado real, contagem confirmada), bem diferente do leito, onde nulo significa "não sei", não "zero confirmado".

Sobre `populacao`, usei `median` em vez de `first`: são só 3 combinações (município, ano) que têm o valor real de população junto com um nulo espúrio na mesma agregação (municípios 352330, 354625, 355120, em 2022 — o bug de 9 linhas já documentado, seção 2.1, ver `.docx`). `median` ignora nulo por padrão e devolve o único valor real que existe, sem precisar de tratamento especial pra esses 3 casos (conferido antes de assumir, não só no achismo).

### 3.1 Anualizando leito e estabelecimento

Decisão (depois de pesquisa adicional antes de bater o martelo, detalhe em `.docx` seção 9): **leito usa a convenção oficial CONASS/DATASUS** (soma dos 12 meses dividida por 12, o mesmo cálculo do indicador nacional de leitos hospitalares por mil habitantes). **Estabelecimento usa média dos meses disponíveis** (não achei convenção oficial de capacidade-tempo equivalente pra contagem de entidade, então a média simples dos meses que existem é o mais razoável).

In [14]:
mensal = df_sp_mun.drop_duplicates(subset=["municipio_6", "ano", "mes"])[
    ["municipio_6", "ano", "mes", "qtd_leitos_existentes", "qtd_estabelecimentos"]
]

por_municipio_ano = mensal.groupby(["municipio_6", "ano"])

# leitos: convenção oficial CONASS/DATASUS (soma dos 12 meses / 12).
# min_count=1 é essencial: sem isso, um grupo com os 12 meses nulos soma "0" por
# padrão do pandas, e viraria "0 leitos" fabricado em vez de nulo de verdade.
leitos_soma_sem_min_count = por_municipio_ano["qtd_leitos_existentes"].sum()
leitos_soma = por_municipio_ano["qtd_leitos_existentes"].sum(min_count=1)
leitos_anual = (leitos_soma / 12).rename("leitos_anual_media")

# estabelecimentos: média dos meses disponíveis (mean já ignora nulo por padrão)
estabelecimentos_anual = por_municipio_ano["qtd_estabelecimentos"].mean().rename("estabelecimentos_anual_media")

anualizado = pd.concat([leitos_anual, estabelecimentos_anual], axis=1).reset_index()

todos_nulos = (leitos_soma_sem_min_count == 0) & (leitos_soma.isna())
total_combinacoes = por_municipio_ano.ngroups
print(f"Combinações município-ano com os 12 meses de leito nulos: {todos_nulos.sum():,} de {total_combinacoes:,} ({todos_nulos.sum()/total_combinacoes:.1%})")
print("Sem min_count=1, essas linhas virariam '0 leitos' por engano.")
anualizado.head()

Combinações município-ano com os 12 meses de leito nulos: 853 de 1,935 (44.1%)
Sem min_count=1, essas linhas virariam '0 leitos' por engano.


,municipio_6,ano,leitos_anual_media,estabelecimentos_anual_media
0,350010,2022,242.833333,153.00
1,350010,2023,233.416667,160.25
2,350010,2024,206.000000,186.75
3,350020,2022,NaN,6.00
4,350020,2023,NaN,5.25


**Achado que vale registrar**: 853 das 1.935 combinações município-ano (44,1%) têm os 12 meses de leito nulos. Sem o `min_count=1`, o `pandas` soma um grupo todo nulo e devolve 0 por padrão (comportamento nada óbvio de quem não conhece essa pegadinha), e quase metade da tabela sairia com um número fabricado.

Isso também responde "por que o valor de leito vai parecer estranho" nos municípios menores: como o gap de cadastro (achado da etapa 2) é maior justamente nos municípios pequenos, a média anual deles fica mais baixa não porque tenham menos leito de verdade, mas porque menos meses foram registrados. Documentado, não corrigido — mesma decisão da etapa 2, agora propagada pro nível anual.

### 3.2 Junta tudo e calcula as 5 métricas centrais

In [15]:
tabela_municipio_ano = metricas_centrais.merge(anualizado, on=["municipio_6", "ano"], how="left")

tabela_municipio_ano["taxa_internacao_uti_10k"] = (
    tabela_municipio_ano["n_internacoes"] / tabela_municipio_ano["populacao"] * 10000
)
tabela_municipio_ano["taxa_icsap_10k"] = (
    tabela_municipio_ano["n_icsap"] / tabela_municipio_ano["populacao"] * 10000
)
tabela_municipio_ano["pct_icsap"] = (
    tabela_municipio_ano["n_icsap"] / tabela_municipio_ano["n_internacoes"] * 100
)
tabela_municipio_ano["custo_medio_internacao"] = (
    tabela_municipio_ano["soma_val_tot"] / tabela_municipio_ano["n_internacoes"]
)
tabela_municipio_ano["dias_medios_uti_internacao"] = (
    tabela_municipio_ano["soma_uti_mes_to"] / tabela_municipio_ano["n_internacoes"]
)

print(f"{len(tabela_municipio_ano):,} linhas, {tabela_municipio_ano.shape[1]} colunas")
tabela_municipio_ano.head()

1,935 linhas, 15 colunas


,municipio_6,ano,n_internacoes,soma_val_tot,soma_uti_mes_to,populacao,nome_municipio,n_icsap,leitos_anual_media,estabelecimentos_anual_media,taxa_internacao_uti_10k,taxa_icsap_10k,pct_icsap,custo_medio_internacao,dias_medios_uti_internacao
0,350010,2022,357,2409622.85,2436,34687.0,Adamantina - SP,81.0,242.833333,153.00,102.920402,23.351688,22.689076,6749.643838,6.823529
1,350010,2023,340,2034558.55,1740,34687.0,Adamantina - SP,61.0,233.416667,160.25,98.019431,17.585839,17.941176,5983.995735,5.117647
2,350010,2024,335,2116637.02,1873,35642.0,Adamantina - SP,59.0,206.000000,186.75,93.990236,16.553504,17.611940,6318.319463,5.591045
3,350020,2022,64,624747.47,381,4351.0,Adolfo - SP,10.0,NaN,6.00,147.092622,22.983222,15.625000,9761.679219,5.953125
4,350020,2023,53,431688.78,242,4351.0,Adolfo - SP,20.0,NaN,5.25,121.811078,45.966444,37.735849,8145.071321,4.566038


As 5 métricas centrais (decisão C, confirmada): taxa de internação com UTI por 10 mil hab, taxa ICSAP por 10 mil hab, % ICSAP, custo médio por internação e dias médios de UTI por internação. Todas ponderadas (soma dividida por soma, nunca média de média), regra 0.1/0.2 aplicada de novo aqui.

### 3.3 Como essas métricas se distribuem entre os municípios

Aqui é onde a regra 0.3 (mediana e IQR, não média e desvio) entra: olhar a distribuição da métrica ENTRE os 645 municípios, não dentro de um município (isso a gente não tem dado pra fazer, ver nota no início da etapa). Usar mediana em vez de média evita que São Paulo capital (extremo grande) puxe a leitura pro lado errado.

In [16]:
metricas_distribuicao = [
    "taxa_internacao_uti_10k", "taxa_icsap_10k", "pct_icsap",
    "custo_medio_internacao", "dias_medios_uti_internacao",
]

distribuicao = tabela_municipio_ano[metricas_distribuicao].describe(percentiles=[0.25, 0.5, 0.75]).T
distribuicao["iqr"] = distribuicao["75%"] - distribuicao["25%"]
distribuicao[["50%", "25%", "75%", "iqr", "min", "max"]].round(2)

,50%,25%,75%,iqr,min,max
taxa_internacao_uti_10k,67.28,50.71,91.44,40.73,11.11,233.67
taxa_icsap_10k,12.05,7.68,19.16,11.47,0.00,64.25
pct_icsap,18.32,14.14,23.53,9.39,0.00,58.33
custo_medio_internacao,7933.72,6726.19,9144.27,2418.08,3463.27,20396.80
dias_medios_uti_internacao,6.35,5.48,7.32,1.84,2.09,17.67


### Leitura da etapa 3

- Tabela município x ano cobre as 1.935 combinações possíveis (645 municípios x 3 anos), sem buraco — todo município de SP teve pelo menos 1 internação com UTI em cada um dos 3 anos.
- Achado técnico que vale registrar: sem o `min_count=1` na soma dos leitos, 44,1% da tabela (853 de 1.935 linhas) sairia com "0 leitos" fabricado em vez de nulo de verdade — pegadinha de comportamento padrão do `pandas`, corrigida antes de virar erro silencioso no dado.
- Mediana entre municípios: 67,3 internações com UTI por 10 mil hab (IQR 40,7), 12,1 ICSAP por 10 mil hab (IQR 11,5), 18,3% de proporção ICSAP (IQR 9,4 pontos percentuais), R$ 7.933,72 de custo médio por internação (IQR R$ 2.418), 6,35 dias médios de UTI por internação (IQR 1,84).
- A mediana de %ICSAP entre municípios (18,3%) bateu bem próxima do agregado geral ponderado pelo total do estado (18,7%, etapa 1) — não precisava bater exato (município grande pesa mais no agregado geral), mas bater perto é um bom sinal de consistência entre os dois jeitos de olhar o mesmo dado.
- Município tem variação grande nas métricas (ex: taxa de internação varia de 11,1 a 233,7 por 10 mil hab) — isso é a entrada natural pra próxima etapa.
- Próxima etapa (4): outliers — quais municípios estão nos extremos dessas distribuições, e se são erro de dado ou achado real (regra 0.4: outlier clínico não se remove cego).

## 4. Outliers (mesma tabela da etapa 3, município x ano)

Regra 0.4 desde o início do notebook: outlier clínico não se remove cego. O objetivo aqui não é "limpar" a tabela, é identificar quem está nos extremos das 5 métricas centrais e decidir, caso a caso, se é ruído (número pequeno de internação, já sabido do etapa 0) ou achado que merece atenção.

Conexão que vale registrar antes de começar (você mesmo notou, cap. 5, seção "Quando tratar um outlier", pág. 183): a primeira diretriz do livro pra outlier é "se deriva claramente de erro, trate como ausente (missing) em vez de manter um valor incorreto". É literalmente a lógica que já usamos na etapa 2, só que ao contrário: lá, em vez de inventar um valor pra um nulo, mantivemos nulo. Mesma filosofia (não fabricar valor duvidoso), aplicada nas duas pontas.

### 4.1 Detectando outlier por IQR (Tukey)

Método clássico de detecção univariada (cap. 5, Análise de Outliers). Fence inferior = Q1 − 1,5×IQR, fence superior = Q3 + 1,5×IQR, aplicado em cada uma das 5 métricas da etapa 3, independente uma da outra.

In [17]:
metricas_distribuicao = [
    "taxa_internacao_uti_10k", "taxa_icsap_10k", "pct_icsap",
    "custo_medio_internacao", "dias_medios_uti_internacao",
]

fences = []
outlier_cols = []
for m in metricas_distribuicao:
    q1, q3 = tabela_municipio_ano[m].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    col = f"outlier_{m}"
    tabela_municipio_ano[col] = (tabela_municipio_ano[m] < lower) | (tabela_municipio_ano[m] > upper)
    outlier_cols.append(col)
    fences.append({
        "metrica": m, "fence_inferior": round(lower, 2), "fence_superior": round(upper, 2),
        "outliers_total": int(tabela_municipio_ano[col].sum()),
        "outliers_baixo": int((tabela_municipio_ano[m] < lower).sum()),
        "outliers_alto": int((tabela_municipio_ano[m] > upper).sum()),
    })

pd.DataFrame(fences)

,metrica,fence_inferior,fence_superior,outliers_total,outliers_baixo,outliers_alto
0,taxa_internacao_uti_10k,-10.39,152.53,40,0,40
1,taxa_icsap_10k,-9.53,36.37,86,0,86
2,pct_icsap,0.04,37.62,40,17,23
3,custo_medio_internacao,3099.07,12771.39,26,0,26
4,dias_medios_uti_internacao,2.73,10.08,47,9,38


`taxa_icsap_10k` é a métrica com mais outlier (86 município-ano), todos pra cima (nenhum fence inferior, porque o mínimo da distribuição já é 0 ICSAP, não dá pra ter menos que isso). `custo_medio_internacao` é a que tem menos (26), e é a única sem nenhum outlier pra baixo — custo muito baixo não é incomum o bastante pra passar a fence.

### 4.2 Quantas métricas cada município-ano estoura ao mesmo tempo

In [18]:
tabela_municipio_ano["n_metricas_outlier"] = tabela_municipio_ano[outlier_cols].sum(axis=1)
tabela_municipio_ano["n_metricas_outlier"].value_counts().sort_index()

n_metricas_outlier
0    1750
1     134
2      48
3       3
Name: count, dtype: int64

1.750 município-ano (90,4%) não estoura fence em nenhuma métrica. Só 3 estouram 3 das 5 ao mesmo tempo, nenhum estoura mais que isso. A maioria do "extremo" é isolado numa métrica só, não um padrão geral de município problemático.

### 4.3 Testando a hipótese: outlier é número pequeno de novo?

A etapa 0 já mostrou que número pequeno de internação deixa taxa instável. Antes de investigar caso a caso, testa se isso também explica boa parte dos outliers daqui (contexto e impacto do outlier, cap. 5, pág. 184).

In [19]:
tem_algum_outlier = tabela_municipio_ano["n_metricas_outlier"] > 0

mediana_n_outlier = tabela_municipio_ano.loc[tem_algum_outlier, "n_internacoes"].median()
mediana_n_nao_outlier = tabela_municipio_ano.loc[~tem_algum_outlier, "n_internacoes"].median()

print(f"Mediana de n_internacoes | é outlier em alguma métrica: {mediana_n_outlier:.0f}")
print(f"Mediana de n_internacoes | não é outlier em nenhuma:    {mediana_n_nao_outlier:.0f}")

Mediana de n_internacoes | é outlier em alguma métrica: 42
Mediana de n_internacoes | não é outlier em nenhuma:    98


Confirma o padrão: quem é outlier tem, em mediana, menos da metade do número de internações de quem não é (42 vs 98). Isso é consistente com a etapa 0 (município com N pequeno tem taxa mais instável), mas **não é a explicação inteira** — como o próximo caso mostra.

### 4.4 O caso que não se explica por número pequeno: Tupã

Estratificar antes de descartar (cap. 5, pág. 184, "considere análise multivariada e estratificação"): olhar um caso específico em vez de só a média do grupo.

In [20]:
colunas_tupa = [
    "nome_municipio", "ano", "populacao", "n_internacoes", "n_icsap",
    "pct_icsap", "taxa_internacao_uti_10k", "taxa_icsap_10k",
]
tabela_municipio_ano[tabela_municipio_ano["nome_municipio"].str.contains("Tupã", na=False)][colunas_tupa]

,nome_municipio,ano,populacao,n_internacoes,n_icsap,pct_icsap,taxa_internacao_uti_10k,taxa_icsap_10k
1848,Tupã - SP,2022,63928.0,956,235.0,24.581590,149.543236,36.760105
1849,Tupã - SP,2023,63928.0,1202,294.0,24.459235,188.024027,45.989238
1850,Tupã - SP,2024,65416.0,1339,291.0,21.732636,204.689984,44.484530


Tupã aparece como outlier de `taxa_icsap_10k` nos 3 anos, e de `taxa_internacao_uti_10k` em 2023 e 2024. Mas tem 956 a 1.339 internações por ano, população de ~64 mil — nada perto do "número pequeno" que explicou os outros casos (ex: Jumirim, 10 internações, Flora Rica, 4). Aqui o outlier é sustentado por volume real, não ruído de amostra pequena. É exatamente o tipo de caso que a regra 0.4 pede pra não remover cego: fica registrado como candidato a investigar na etapa 5 (correlação), pode ter relação real com a estrutura de saúde local do município, não erro de dado.

### 4.5 Decisão de tratamento

Nenhuma linha foi removida da tabela. Outlier aqui vira nota, não exclusão (regra 0.4). Resumo do que foi feito com cada grupo:

| Situação | O que fizemos |
|---|---|
| Outlier isolado (1 métrica), N pequeno de internação | Explicado por ruído de amostra pequena (etapa 0), mantido, não investigado caso a caso individualmente |
| Outlier com N grande (ex: Tupã) | Mantido, sinalizado como candidato a investigação real na etapa 5 |
| Qualquer outlier | Nenhuma remoção de linha, nenhuma substituição de valor |

O item 6 do livro (pág. 184) diz pra consultar especialista do domínio antes de decidir. Não temos esse recurso disponível agora (desculpa, Nemec — não temos nenhum parente médico no time), então a checagem que dá pra fazer sem especialista é a de contexto quantitativo acima (tamanho de amostra, consistência entre anos), documentada em vez de decidida no escuro.

### Leitura da etapa 4

- 90,4% dos município-ano não é outlier em nenhuma das 5 métricas. Extremos existem, mas não são a maioria.
- Boa parte do que é outlier se explica pelo mesmo problema já mapeado na etapa 0 (N pequeno de internação deixa taxa instável) — mediana de 42 internações em quem é outlier, contra 98 em quem não é.
- Nem tudo se explica assim: Tupã é outlier de taxa ICSAP nos 3 anos com volume grande (956 a 1.339 internações/ano), candidato real a olhar de novo na etapa 5 (correlação), não um artefato de amostra pequena.
- Nenhuma linha foi removida ou alterada nessa etapa — outlier virou nota documentada, não exclusão.
- Próxima etapa (5): correlação — será que taxa de internação/ICSAP se relaciona com capacidade instalada (leito, estabelecimento) por município?

## 5. Correlação (unidade: município x ano)

Pergunta: taxa de internação/ICSAP se relaciona com capacidade instalada (leito, estabelecimento) do município? Mesma tabela das etapas 3 e 4.

Método (decidido antes de rodar): **Spearman como principal**, Pearson como comparação secundária, sem Kendall. Motivo: Pearson assume relação linear e é sensível a outlier/assimetria, e a etapa 4 acabou de confirmar outlier real nas métricas centrais (inclusive um caso de volume grande, Tupã). Spearman (baseado em ranking) é mais robusto a isso, e segue a mesma lógica da regra 0.3 (mediana/IQR em vez de média/desvio).

In [21]:
colunas_correlacao = metricas_distribuicao + ["leitos_anual_media", "estabelecimentos_anual_media"]

n_leito_valido = tabela_municipio_ano["leitos_anual_media"].notna().sum()
print(f"Correlações com leitos_anual_media usam {n_leito_valido:,} de {len(tabela_municipio_ano):,} linhas "
      f"({n_leito_valido/len(tabela_municipio_ano):.1%}) — o resto é nulo (achado da etapa 3, gap CNES-LT).")
print(f"Correlações com estabelecimentos_anual_media usam {tabela_municipio_ano['estabelecimentos_anual_media'].notna().sum():,} linhas (sem nulo relevante).")

correlacao_spearman = tabela_municipio_ano[colunas_correlacao].corr(method="spearman")
correlacao_pearson = tabela_municipio_ano[colunas_correlacao].corr(method="pearson")

correlacao_spearman.loc[metricas_distribuicao, ["leitos_anual_media", "estabelecimentos_anual_media"]].round(3)

Correlações com leitos_anual_media usam 1,082 de 1,935 linhas (55.9%) — o resto é nulo (achado da etapa 3, gap CNES-LT).
Correlações com estabelecimentos_anual_media usam 1,935 linhas (sem nulo relevante).


,leitos_anual_media,estabelecimentos_anual_media
taxa_internacao_uti_10k,0.016,-0.240
taxa_icsap_10k,0.090,-0.120
pct_icsap,0.145,0.017
custo_medio_internacao,-0.092,-0.013
dias_medios_uti_internacao,0.120,0.207


**Leitura honesta**: nenhuma correlação forte. A mais alta (em valor absoluto) é -0,24, entre `taxa_internacao_uti_10k` e `estabelecimentos_anual_media` — fraca, e negativa (mais estabelecimento de saúde no município associado a taxa de internação com UTI um pouco menor, possível sinal de acesso a atenção primária, mas fraco demais pra afirmar isso com confiança). `dias_medios_uti_internacao` tem a segunda maior, 0,21 com estabelecimento (positiva). Tudo o mais fica abaixo de 0,15.

Isso também é um achado: **capacidade instalada sozinha não explica bem a variação dessas métricas entre municípios**. Se houver relação real, ela deve depender de outra coisa (porte, região, perfil socioeconômico) que não está nessa tabela.

In [22]:
diferenca_metodo = (
    correlacao_spearman.loc[metricas_distribuicao, ["leitos_anual_media", "estabelecimentos_anual_media"]]
    - correlacao_pearson.loc[metricas_distribuicao, ["leitos_anual_media", "estabelecimentos_anual_media"]]
).abs()

print(f"Maior diferença absoluta entre Spearman e Pearson: {diferenca_metodo.values.max():.3f}")
diferenca_metodo.round(3)

Maior diferença absoluta entre Spearman e Pearson: 0.186


,leitos_anual_media,estabelecimentos_anual_media
taxa_internacao_uti_10k,0.052,0.186
taxa_icsap_10k,0.123,0.080
pct_icsap,0.159,0.028
custo_medio_internacao,0.114,0.018
dias_medios_uti_internacao,0.075,0.169


A diferença chega a 0,19 num dos casos — Spearman e Pearson não contam a mesma história aqui, o que valida a preocupação de método: se tivéssemos usado só Pearson (sensível a outlier/assimetria), a leitura teria sido diferente, possivelmente escondendo ou inflando uma relação que não é bem assim. Reforça que a escolha de Spearman como principal foi acertada pra esse dado, não só teoricamente.

### 5.1 Revisitando Tupã com uma comparação justa (por habitante, não valor bruto)

`leitos_anual_media` bruto favorece município grande. Pra comparar Tupã com o resto de forma justa, olha leito por 10 mil habitantes, não o número absoluto.

In [23]:
tabela_municipio_ano["leitos_10k_hab"] = (
    tabela_municipio_ano["leitos_anual_media"] / tabela_municipio_ano["populacao"] * 10000
)

mediana_leitos_10k = tabela_municipio_ano["leitos_10k_hab"].median()
tupa_leitos_10k = tabela_municipio_ano[tabela_municipio_ano["nome_municipio"].str.contains("Tupã", na=False)][
    ["ano", "leitos_10k_hab", "taxa_icsap_10k"]
]

print(f"Mediana geral de leitos por 10 mil hab: {mediana_leitos_10k:.1f}")
tupa_leitos_10k

Mediana geral de leitos por 10 mil hab: 21.2


,ano,leitos_10k_hab,taxa_icsap_10k
1848,2022,59.389730,36.760105
1849,2023,58.190464,45.989238
1850,2024,57.019689,44.484530


Tupã tem entre 57 e 59 leitos por 10 mil hab, quase **3x a mediana geral** (21,2). Isso muda a leitura do achado da etapa 4: a taxa de ICSAP alta de Tupã não parece erro nem sinal isolado, ela acompanha uma capacidade instalada muito acima do típico pro tamanho do município (população ~64 mil, tamanho médio, mas leito de cidade bem maior).

Hipótese mais provável (não provada aqui, registrada como hipótese): Tupã funciona como polo de saúde regional (recebe paciente de município vizinho menor), o que infla tanto a capacidade instalada quanto o volume de internação registrado ali — não é necessariamente que Tupã "adoece mais", pode ser que Tupã "atende mais gente que não é só de Tupã". Isso não dá pra confirmar só com esse dataset (não temos município de origem do paciente dentro de SP, só a UF), fica como pergunta em aberto pra quem for aprofundar depois.

### Leitura da etapa 5

- Nenhuma correlação forte entre capacidade instalada (leito, estabelecimento) e as métricas centrais — a mais alta é -0,24 (taxa de internação x estabelecimento), o resto abaixo de 0,21. Achado honesto: capacidade instalada sozinha não explica a variação entre municípios.
- Spearman e Pearson divergem até 0,19 num dos casos — confirma que a escolha de método (Spearman, robusto a outlier/assimetria) não foi só precaução teórica, fez diferença real na leitura.
- O caso Tupã (etapa 4) ganhou uma explicação plausível: leito por 10 mil hab quase 3x a mediana geral, hipótese de polo regional de saúde — não confirmável com esse dataset, fica registrado como pergunta em aberto.
- Próxima etapa (6): sazonalidade, unidade mês x estado inteiro (decidida na etapa 0), não por município.

## 6. Sazonalidade (unidade: mês x estado inteiro)

Unidade decidida desde a etapa 0: mês x ESTADO INTEIRO, não por município — comparar município por mês teria o mesmo problema de número pequeno que já vimos nas etapas anteriores (e pior).

Diferença importante em relação às etapas 3 e 4 (nível município): lá excluímos `reside_fora_sp`, porque paciente de fora de SP não tem população de município de SP pra virar denominador de taxa. Aqui, por decisão do Vitor, **não excluímos** — o objetivo agora é olhar a carga total que o sistema de saúde de SP absorve por mês, residente ou não, porque isso também mede o quanto custa pra SP cuidar desses pacientes (inclusive os de fora). Resolução técnica: o NUMERADOR das métricas usa todos os atendimentos ocorridos em SP; o DENOMINADOR populacional (nas duas taxas) continua sendo só a população dos municípios de SP — não faria sentido inflar população com gente que mora fora. Isso significa que as taxas aqui não são uma "taxa de risco pra quem mora em SP" pura, são mais uma "carga assistencial por 10 mil habitantes de SP", que já embute o efeito de SP ser polo de referência.

In [24]:
import calendar

pop_dim_sp = (
    df[~df["reside_fora_sp"]]
    .drop_duplicates(subset=["municipio_6", "ano"])[["municipio_6", "ano", "populacao"]]
)
populacao_sp_ano = pop_dim_sp.groupby("ano")["populacao"].sum()

tabela_ano_mes = (
    df.groupby(["ano", "mes"])
    .agg(
        n_internacoes=("qtd_internacoes_uti", "sum"),
        soma_val_tot=("soma_val_tot", "sum"),
        soma_uti_mes_to=("soma_uti_mes_to", "sum"),
    )
    .reset_index()
)

n_icsap_mes = (
    df[df["csap"] == "sim"].groupby(["ano", "mes"])["qtd_internacoes_uti"].sum().rename("n_icsap")
)
tabela_ano_mes = tabela_ano_mes.merge(n_icsap_mes, on=["ano", "mes"], how="left")
tabela_ano_mes["n_icsap"] = tabela_ano_mes["n_icsap"].fillna(0)

tabela_ano_mes = tabela_ano_mes.merge(populacao_sp_ano.rename("populacao_sp"), on="ano", how="left")

tabela_ano_mes["taxa_internacao_uti_10k"] = tabela_ano_mes["n_internacoes"] / tabela_ano_mes["populacao_sp"] * 10000
tabela_ano_mes["taxa_icsap_10k"] = tabela_ano_mes["n_icsap"] / tabela_ano_mes["populacao_sp"] * 10000
tabela_ano_mes["pct_icsap"] = tabela_ano_mes["n_icsap"] / tabela_ano_mes["n_internacoes"] * 100
tabela_ano_mes["custo_medio_internacao"] = tabela_ano_mes["soma_val_tot"] / tabela_ano_mes["n_internacoes"]
tabela_ano_mes["dias_medios_uti_internacao"] = tabela_ano_mes["soma_uti_mes_to"] / tabela_ano_mes["n_internacoes"]

metricas_sazonais = ["taxa_internacao_uti_10k", "taxa_icsap_10k", "pct_icsap", "custo_medio_internacao", "dias_medios_uti_internacao"]

print(f"Tabela ano+mês: {tabela_ano_mes.shape[0]} linhas (esperado 36 = 3 anos x 12 meses).")
tabela_ano_mes[["ano", "mes"] + metricas_sazonais].round(2)

Tabela ano+mês: 36 linhas (esperado 36 = 3 anos x 12 meses).


,ano,mes,taxa_internacao_uti_10k,taxa_icsap_10k,pct_icsap,custo_medio_internacao,dias_medios_uti_internacao
0,2022,1,4.79,0.80,16.72,7811.55,7.24
1,2022,2,4.39,0.65,14.74,8721.79,7.46
2,2022,3,4.67,0.77,16.51,7838.84,7.18
3,2022,4,4.55,0.85,18.62,7849.55,7.02
4,2022,5,4.65,0.89,19.22,7789.39,6.92
5,2022,6,4.50,0.84,18.56,7662.70,6.86
6,2022,7,4.97,0.94,18.83,7930.83,7.18
7,2022,8,4.84,0.90,18.69,7929.25,7.06
8,2022,9,4.76,0.91,19.15,8108.75,7.06
9,2022,10,4.97,0.96,19.36,8091.38,7.06


**Leitura da série temporal**: dá pra ver a tendência de crescimento (mesma da etapa 1, ICSAP subindo ano a ano) misturada com qualquer variação dentro do próprio ano. Pra separar isso da sazonalidade "pura", precisa agrupar por mês do calendário (janeiro, fevereiro...), não por ano+mês.

In [25]:
tabela_ano_mes["dias_no_mes"] = tabela_ano_mes.apply(
    lambda r: calendar.monthrange(int(r["ano"]), int(r["mes"]))[1], axis=1
)

padrao_sazonal = tabela_ano_mes.groupby("mes")[metricas_sazonais].median()
padrao_sazonal["n_anos"] = tabela_ano_mes.groupby("mes").size()

print("Padrão sazonal por mês (mediana dos 3 anos, N=3 por mês — leitura exploratória, não teste estatístico):")
padrao_sazonal.round(3)

Padrão sazonal por mês (mediana dos 3 anos, N=3 por mês — leitura exploratória, não teste estatístico):


,taxa_internacao_uti_10k,taxa_icsap_10k,pct_icsap,custo_medio_internacao,dias_medios_uti_internacao,n_anos
mes,,,,,,
1,5.003,0.951,18.595,7893.275,6.990,3
2,4.735,0.885,17.833,8059.315,6.852,3
3,5.243,0.991,18.318,7838.838,6.718,3
4,4.955,0.927,18.708,7849.548,6.767,3
5,5.095,0.926,19.219,7789.388,6.783,3
6,5.203,0.948,18.556,7942.005,6.860,3
7,5.223,0.966,18.825,8065.992,6.977,3
8,5.423,1.044,19.246,8065.596,6.848,3
9,5.008,0.942,18.810,8108.752,6.684,3


**Aviso**: N=3 por mês (um valor por ano) é pouquíssimo pra qualquer teste estatístico sério — tratamos isso como leitura exploratória, mostrando também min/máx, não como conclusão fechada.

**Achado à parte, antes de interpretar sazonalidade**: mês não tem o mesmo número de dias (fevereiro tem 28, os outros têm 30 ou 31). Isso sozinho já faz fevereiro parecer "mais baixo" sem ser sazonalidade de verdade, é só ter menos dias pra internação acontecer. Corrigindo por dia:

In [26]:
tabela_ano_mes["taxa_internacao_uti_10k_dia"] = tabela_ano_mes["taxa_internacao_uti_10k"] / tabela_ano_mes["dias_no_mes"]
tabela_ano_mes["taxa_icsap_10k_dia"] = tabela_ano_mes["taxa_icsap_10k"] / tabela_ano_mes["dias_no_mes"]

padrao_sazonal_dia = tabela_ano_mes.groupby("mes")[["taxa_internacao_uti_10k_dia", "taxa_icsap_10k_dia"]].median()
variacao_pct = (padrao_sazonal_dia.max() - padrao_sazonal_dia.min()) / padrao_sazonal_dia.median() * 100

print("Taxa por DIA (corrige o viés de mês com menos dias, ex: fevereiro):")
print(padrao_sazonal_dia.sort_values("taxa_internacao_uti_10k_dia").round(4))
print()
print("Variação (mês mais alto - mês mais baixo) / mediana:")
print(variacao_pct.round(1))

Taxa por DIA (corrige o viés de mês com menos dias, ex: fevereiro):
     taxa_internacao_uti_10k_dia  taxa_icsap_10k_dia
mes                                                 
1                         0.1614              0.0307
5                         0.1644              0.0299
4                         0.1652              0.0309
11                        0.1655              0.0315
10                        0.1660              0.0323
12                        0.1660              0.0305
9                         0.1669              0.0314
7                         0.1685              0.0312
2                         0.1691              0.0305
3                         0.1691              0.0320
6                         0.1734              0.0316
8                         0.1749              0.0337

Variação (mês mais alto - mês mais baixo) / mediana:
taxa_internacao_uti_10k_dia     8.1
taxa_icsap_10k_dia             12.1
dtype: float64


**Leitura honesta**: depois de corrigir por dia, fevereiro deixa de ser o mês "mais baixo" (era viés de calendário). A variação real fica em torno de 8-9%, com junho/agosto (inverno) um pouco acima e janeiro/maio um pouco abaixo — sinal fraco no agregado, mas existe uma leve tendência de inverno puxando mais internação. Vale abrir por grupo de diagnóstico antes de descartar sazonalidade: o agregado pode estar escondendo sinais que se cancelam.

### 6.1 Quebra por grupo CSAP: o agregado esconde sinal mais forte

Grupos: os 3 principais ICSAP da etapa 1 (`g12` Cerebrovasculares, `g11` Insuf. cardíaca, `g10` Angina) mais um grupo de comparação (`g08` Pulmonares) — não é ICSAP top 3, mas é o marcador clássico de sazonalidade respiratória na literatura, serve de referência pra saber se o método está captando sinal de verdade.

In [27]:
grupos_sazonalidade = {"g12": "Cerebrovasculares", "g11": "Insuf. cardíaca", "g10": "Angina", "g08": "Pulmonares"}

sub_grupos = df[df["grupo_csap_cod"].isin(grupos_sazonalidade)].copy()
tabela_grupos_mes = (
    sub_grupos.groupby(["grupo_csap_cod", "ano", "mes"])["qtd_internacoes_uti"].sum().reset_index()
)
tabela_grupos_mes["dias_no_mes"] = tabela_grupos_mes.apply(
    lambda r: calendar.monthrange(int(r["ano"]), int(r["mes"]))[1], axis=1
)
tabela_grupos_mes["internacoes_dia"] = tabela_grupos_mes["qtd_internacoes_uti"] / tabela_grupos_mes["dias_no_mes"]

resumo_grupos = []
for cod, nome in grupos_sazonalidade.items():
    pool_grupo = tabela_grupos_mes[tabela_grupos_mes["grupo_csap_cod"] == cod].groupby("mes")["internacoes_dia"].median()
    variacao = (pool_grupo.max() - pool_grupo.min()) / pool_grupo.median() * 100
    resumo_grupos.append({
        "grupo": f"{cod} ({nome})",
        "mes_mais_baixo": int(pool_grupo.idxmin()),
        "mes_mais_alto": int(pool_grupo.idxmax()),
        "variacao_pct": round(variacao, 1),
    })

resumo_grupos_df = pd.DataFrame(resumo_grupos).sort_values("variacao_pct")
resumo_grupos_df

,grupo,mes_mais_baixo,mes_mais_alto,variacao_pct
0,g12 (Cerebrovasculares),5,8,14.2
1,g11 (Insuf. cardíaca),4,9,19.6
2,g10 (Angina),5,2,24.6
3,g08 (Pulmonares),2,5,67.9


**Confronto com a hipótese do Vitor** ("ataque cardíaco no calor"): os dados não sustentam essa direção. Angina (`g10`) até tem pico num mês de verão, mas é o segundo sinal mais fraco dos quatro (N=3 por mês, cuidado pra não superinterpretar). Insuficiência cardíaca (`g11`) puxa mais pro inverno, e pulmonares (`g08`) — de longe o sinal mais forte, quase 70% de variação — é bem mais alto no outono/inverno, o oposto de calor.

Isso bate com literatura real sobre o assunto, não é só achado deste dataset: um estudo específico sobre a cidade de São Paulo (Silva et al., *PLOS ONE*, 2018) encontrou aumento de internação por insuficiência cardíaca descompensada e infarto agudo do miocárdio durante o inverno paulistano (mesmo sendo um inverno ameno), não no calor. E doença respiratória aumentando internação no inverno é padrão bem estabelecido na literatura brasileira (Jornal da USP, entre outros).

Correção honesta da hipótese inicial: não é calor que explica o padrão cardiovascular aqui, é frio/inverno — mesma direção do padrão respiratório, só que mais fraco e mais difícil de separar de ruído com N=3 por mês. Fontes externas usadas nesta parte abaixo.

### Leitura da etapa 6

- Tabela ano+mês (36 linhas, nível estado inteiro): decisão nova do Vitor, numerador inclui todo mundo atendido em SP (residente ou não, pra medir a carga real do sistema), denominador populacional continua só residente de SP.
- Sinal agregado de sazonalidade é fraco (~8-9% de variação) depois de corrigir o viés de mês com menos dias (fevereiro).
- Quebrado por grupo CSAP o sinal fica mais claro: pulmonar (quase 70% de variação, mais alto no outono/inverno) é o exemplo mais forte de sazonalidade real; insuficiência cardíaca também pesa mais pro inverno; angina teve pico de verão, mas é sinal fraco e com N pequeno.
- Hipótese inicial do Vitor ("calor" pro cardiovascular) não bate nem com os dados nem com a literatura específica de São Paulo (PLOS ONE, 2018) — é frio que pesa mais, mesma direção do respiratório.
- Próxima etapa (7): recorte geográfico.

**Fontes externas usadas nesta etapa**:
- Silva et al., "Increased hospitalizations for decompensated heart failure and acute myocardial infarction during mild winters: A seven-year experience in the public health system of the largest city in Latin America", PLOS ONE, 2018 — https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0190733
- Jornal da USP, "Doenças respiratórias aumentam internações hospitalares no inverno" — https://jornal.usp.br/atualidades/doencas-respiratorias-aumentam-internacoes-hospitalares-no-inverno/

## 7. Recorte geográfico (unidade: município x ano, volta ao nível das etapas 3-5)

Duas partes:

**7.1** revisar a hipótese "polo de saúde regional" (etapa 5, caso Tupã) de forma sistemática, não só o caso isolado.

**7.2** juntar uma dimensão de região geográfica (Região Intermediária do IBGE) que o dataset não tinha até agora, pra ver se a variação entre município tem padrão espacial. Fonte: classificação oficial do IBGE (API `servicodados.ibge.gov.br`), baixada pelo Vitor no ambiente dele — decisão dele de usar a fonte mais robusta (região oficial do IBGE) em vez de uma simplificação RMSP x Interior.

### 7.1 Revisitando "polo de saúde regional" em todos os municípios

Etapa 5 achou que Tupã tinha leito por 10 mil hab quase 3x a mediana geral, e levantou a hipótese de polo regional. Primeiro passo: ver o ranking completo, não só o caso Tupã.

In [28]:
ranking_leitos = (
    tabela_municipio_ano.groupby(["municipio_6", "nome_municipio"])["leitos_10k_hab"]
    .median()
    .reset_index()
    .sort_values("leitos_10k_hab", ascending=False)
)

mediana_geral_leitos = ranking_leitos["leitos_10k_hab"].median()
print(f"Mediana geral de leitos por 10 mil hab: {mediana_geral_leitos:.1f}")
print(f"Total de municípios no ranking: {len(ranking_leitos)}")
ranking_leitos.head(15).reset_index(drop=True)

Mediana geral de leitos por 10 mil hab: 21.3
Total de municípios no ranking: 645


,municipio_6,nome_municipio,leitos_10k_hab
0,352450,Jaci - SP,283.725207
1,351390,Divinolândia - SP,166.696541
2,351080,Casa Branca - SP,144.571449
3,352260,Itapira - SP,132.181833
4,353260,Nhandeara - SP,130.937881
5,353890,Pirajuí - SP,128.728100
6,354750,Santa Rita do Passa Quatro - SP,117.954603
7,351518,Espírito Santo do Pinhal - SP,110.696705
8,353620,Pariquera-Açu - SP,102.168148
9,350290,Araçoiaba da Serra - SP,101.716857


**Leitura honesta**: o ranking bruto não confirma a hipótese como estava. Tupã nem aparece no topo — os primeiros lugares são município bem menores (ex: Jaci, população 7.613, "apenas" 216 leitos rende um `leitos_10k_hab` de quase 284). Isso cheira a ruído de denominador pequeno, o mesmo problema de número pequeno que já vimos nas etapas 0, 4 e 6, só que agora afetando uma taxa de capacidade, não de internação. Antes de aceitar ou descartar a hipótese, precisa checar se `leitos_10k_hab` alto realmente anda junto de volume real de atendimento, ou se é só ruído.

In [29]:
resumo_municipio = (
    tabela_municipio_ano.groupby(["municipio_6", "nome_municipio"])
    .agg(leitos_10k_hab=("leitos_10k_hab", "median"), n_internacoes=("n_internacoes", "median"), populacao=("populacao", "median"))
    .dropna()
    .reset_index()
)

corr_leito_volume = resumo_municipio[["leitos_10k_hab", "n_internacoes"]].corr(method="spearman").iloc[0, 1]
corr_leito_populacao = resumo_municipio[["leitos_10k_hab", "populacao"]].corr(method="spearman").iloc[0, 1]

print(f"N de municípios com leito válido nos 3 anos: {len(resumo_municipio)} de {tabela_municipio_ano['municipio_6'].nunique()}")
print(f"Correlação (Spearman) leitos_10k_hab x n_internacoes: {corr_leito_volume:.3f}")
print(f"Correlação (Spearman) leitos_10k_hab x população: {corr_leito_populacao:.3f}")

N de municípios com leito válido nos 3 anos: 364 de 645
Correlação (Spearman) leitos_10k_hab x n_internacoes: -0.012
Correlação (Spearman) leitos_10k_hab x população: -0.120


Confirmado: `leitos_10k_hab` sozinho quase não se relaciona com volume real de internação nem com população (as duas correlações ficam perto de zero). O ranking bruto é dominado por ruído de município pequeno — não é isso que caracteriza um "polo regional" de verdade. Um polo regional de verdade precisa das duas coisas juntas: capacidade alta POR HABITANTE **e** volume absoluto de atendimento alto (sinal de que atende gente além da própria população). Refazendo o filtro com esse critério:

In [30]:
p75_leito = resumo_municipio["leitos_10k_hab"].quantile(0.75)
p75_volume = resumo_municipio["n_internacoes"].quantile(0.75)

candidatos_polo = resumo_municipio[
    (resumo_municipio["leitos_10k_hab"] >= p75_leito) & (resumo_municipio["n_internacoes"] >= p75_volume)
].sort_values("n_internacoes", ascending=False)

print(f"Candidatos a polo regional (top 25% em leito per capita E top 25% em volume): {len(candidatos_polo)} de {len(resumo_municipio)} municípios")
candidatos_polo.round(1).reset_index(drop=True)

Candidatos a polo regional (top 25% em leito per capita E top 25% em volume): 20 de 364 municípios


,municipio_6,nome_municipio,leitos_10k_hab,n_internacoes,populacao
0,355030,São Paulo - SP,33.0,60093.0,11451999.0
1,350950,Campinas - SP,31.0,6865.0,1139047.0
2,354340,Ribeirão Preto - SP,38.5,3771.0,698642.0
3,354980,São José do Rio Preto - SP,53.7,3570.0,480393.0
4,354850,Santos - SP,47.5,3183.0,418608.0
5,350600,Bauru - SP,41.8,2610.0,379146.0
6,352900,Marília - SP,43.2,2448.0,237627.0
7,354880,São Caetano do Sul - SP,59.7,2030.0,165655.0
8,352530,Jaú - SP,78.5,1396.0,133497.0
9,350550,Barretos - SP,50.0,1281.0,122485.0


Agora sim: a lista bate com cidades conhecidas como polo regional de saúde no interior de SP (São José do Rio Preto, Ribeirão Preto, Marília, Bauru, Botucatu, Presidente Prudente, Catanduva, Jaú...), mais os grandes centros óbvios (capital, Campinas, Santos). **Tupã está na lista** (13º lugar em volume entre os 20 candidatos), o que dá mais sustentação à hipótese da etapa 5 — mas agora sabemos que Tupã não é um caso isolado, é um exemplo de um padrão que se repete em ~20 município do estado.

Ressalva: essa lista vem de um subconjunto (municípios com leito válido nos 3 anos) — o mesmo gap de cobertura CNES-LT documentado na etapa 2 reduz quem entra nessa conta, então pode haver polo regional real "escondido" atrás de dado faltante.

### 7.2 Recorte por Região Intermediária (IBGE)

Junta a classificação oficial do IBGE (Região Geográfica Intermediária, 11 regiões cobrindo os 645 municípios de SP) na tabela de métricas centrais, pra ver se região explica parte da variação que capacidade instalada (etapa 5) não explicou.

In [31]:
regiao_municipios = pd.read_csv("dados/regiao_municipios_sp.csv", dtype={"municipio_6": str})

tabela_municipio_ano_regiao = tabela_municipio_ano.merge(
    regiao_municipios[["municipio_6", "regiao_intermediaria"]], on="municipio_6", how="left"
)

print(f"Municípios sem região identificada: {tabela_municipio_ano_regiao['regiao_intermediaria'].isna().sum()} de {len(tabela_municipio_ano_regiao)} linhas.")
print(f"Regiões intermediárias cobrindo SP: {tabela_municipio_ano_regiao['regiao_intermediaria'].nunique()}")

Municípios sem região identificada: 0 de 1935 linhas.
Regiões intermediárias cobrindo SP: 11


In [32]:
metricas_por_regiao = (
    tabela_municipio_ano_regiao.groupby("regiao_intermediaria")[metricas_distribuicao].median().round(2)
)
n_municipios_regiao = regiao_municipios.groupby("regiao_intermediaria").size().rename("n_municipios")
metricas_por_regiao = metricas_por_regiao.merge(n_municipios_regiao, on="regiao_intermediaria")

metricas_por_regiao.sort_values("taxa_icsap_10k", ascending=False)

,taxa_internacao_uti_10k,taxa_icsap_10k,pct_icsap,custo_medio_internacao,dias_medios_uti_internacao,n_municipios
regiao_intermediaria,,,,,,
São José do Rio Preto,99.42,17.41,18.83,9054.57,5.81,100
Marília,90.77,17.40,19.91,7649.95,6.35,54
São José dos Campos,75.04,15.28,20.50,7398.70,6.78,39
Araraquara,69.44,15.23,22.17,6671.43,5.51,26
Araçatuba,70.01,14.00,22.59,7101.18,5.75,44
Presidente Prudente,63.18,12.17,20.00,7644.17,6.08,55
Bauru,73.74,12.00,16.76,7183.66,6.42,48
Ribeirão Preto,63.15,11.23,19.05,8496.09,5.96,64
Campinas,57.03,9.36,16.90,7961.66,6.46,87


**Achado principal da etapa 7**: aqui sim tem um padrão espacial forte — bem mais forte que qualquer coisa que vimos até agora (mais forte que capacidade instalada na etapa 5, mais forte que sazonalidade na etapa 6). `taxa_icsap_10k` mediana varia de 7,65 (Sorocaba) a 17,41 (São José do Rio Preto), mais que o dobro. `taxa_internacao_uti_10k` varia de 48,3 a 99,4, também mais que o dobro.

Vale notar pra tese central do projeto: a região que inclui a capital (São Paulo) tem uma das taxas de ICSAP mais baixas (8,48), atrás só de Sorocaba (7,65). É consistente com a hipótese de que região com melhor cobertura de atenção primária interna menos por condição evitável — mas é só consistência, não prova: não temos dado de cobertura de atenção primária (SIA) neste dataset, ficou fora do escopo desde o início.

In [33]:
dispersao_regiao = (
    tabela_municipio_ano_regiao.groupby("regiao_intermediaria")["taxa_icsap_10k"]
    .apply(lambda s: s.quantile(0.75) - s.quantile(0.25))
    .rename("iqr_taxa_icsap_10k")
    .sort_values(ascending=False)
)

print("IQR de taxa_icsap_10k por região (dispersão interna, comparar com o IQR geral da etapa 3: 11,5):")
dispersao_regiao.round(2)

IQR de taxa_icsap_10k por região (dispersão interna, comparar com o IQR geral da etapa 3: 11,5):


regiao_intermediaria
São José do Rio Preto    17.51
Marília                  15.08
Presidente Prudente      14.84
Araçatuba                14.16
São José dos Campos      11.51
Araraquara                9.75
Bauru                     9.00
Ribeirão Preto            8.07
Campinas                  7.33
Sorocaba                  6.59
São Paulo                 4.51
Name: iqr_taxa_icsap_10k, dtype: float64

A dispersão interna também varia por região: São Paulo é a mais homogênea (IQR 4,51, bem abaixo do IQR geral de 11,5 da etapa 3), enquanto São José do Rio Preto e Marília são as mais heterogêneas (IQR acima de 15) — mesmo sendo as regiões com taxa mediana mais alta, elas têm bastante município discrepante lá dentro também, não é um bloco uniforme.

### Leitura da etapa 7

- Ranking bruto de leito por habitante é dominado por ruído de município pequeno (correlação com volume real ≈ 0); refinado pra "top 25% em capacidade E top 25% em volume", aparecem ~20 município reconhecíveis como polo regional de saúde (interior + grandes centros), Tupã entre eles — a hipótese da etapa 5 ganha sustentação como padrão, não caso isolado.
- Região Intermediária (IBGE) é o fator geográfico mais explicativo até agora: `taxa_icsap_10k` mediana varia mais que o dobro entre região (7,65 a 17,41), mais forte que capacidade instalada (etapa 5) ou mês (etapa 6).
- Região da capital (São Paulo) e Sorocaba têm as taxas de ICSAP medianas mais baixas — consistente (não prova) com a tese central do projeto de que atenção primária melhor reduz internação evitável.
- Dispersão interna varia por região; São Paulo é a mais homogênea, São José do Rio Preto e Marília as mais heterogêneas.
- Próxima etapa (8): síntese final.

## 8. Síntese final da AED

Fecha as 7 etapas anteriores da Análise Exploratória de Dados sobre o `dataset_unificado_sp_2022_2024.parquet` (SIH-UTI classificado por ICSAP, 645 municípios de SP, 2022-2024, 463.299 linhas agregadas, 813.422 internações reais).

**Recapitulação:**

- **Etapa 1 (perfil geral)**: 18,7% das internações com UTI são ICSAP (evitáveis em tese, se a atenção primária tivesse funcionado antes). Top 3 grupos: cerebrovascular, insuficiência cardíaca, angina.
- **Etapa 2 (missing)**: dois padrões de nulo, os dois MAR (não MCAR) e documentados; nenhuma imputação feita nesta AED, mantido nulo e explicado.
- **Etapa 3 (distribuição)**: 5 métricas centrais por município x ano, sempre ponderadas por `qtd_internacoes_uti` e sempre lidas por mediana/IQR (nunca média/desvio).
- **Etapa 4 (outliers)**: outlier em geral é número pequeno, mas não sempre — Tupã-SP é outlier real de taxa ICSAP, com volume grande, não ruído de amostra pequena.
- **Etapa 5 (correlação)**: capacidade instalada (leito, estabelecimento) não explica bem a variação entre município (correlação fraca, no máximo -0,24). Caso Tupã revisitado: leito por 10 mil hab quase 3x a mediana geral, hipótese de polo regional.
- **Etapa 6 (sazonalidade)**: sinal agregado fraco (~8-9% de variação mês a mês), mas forte quando quebrado por grupo CSAP (pulmonar varia quase 70%, mais alto no outono/inverno).
- **Etapa 7 (recorte geográfico)**: hipótese de polo regional confirmada de forma sistemática (Tupã aparece entre os 20 municípios candidatos); região geográfica (IBGE) é o fator mais explicativo encontrado até agora, taxa ICSAP mediana varia mais que o dobro entre região.

### 8.1 Tabela resumo: as 5 métricas centrais em cada etapa

Junta, pra cada uma das 5 métricas centrais, o que cada etapa encontrou: mediana geral (etapa 3), % de município-ano outlier (etapa 4), correlação com capacidade instalada (etapa 5), variação sazonal (etapa 6, só existe pras duas taxas) e variação entre região (etapa 7).

In [34]:
fences_df = pd.DataFrame(fences).set_index("metrica")

resumo_etapas = pd.DataFrame({
    "mediana_geral": distribuicao["50%"],
    "iqr": distribuicao["iqr"],
})
resumo_etapas["pct_outlier"] = (fences_df["outliers_total"] / len(tabela_municipio_ano) * 100).round(1)
resumo_etapas["corr_spearman_c_estabelecimento"] = correlacao_spearman.loc[
    metricas_distribuicao, "estabelecimentos_anual_media"
].round(3)

variacao_sazonal_map = {
    "taxa_internacao_uti_10k": variacao_pct.get("taxa_internacao_uti_10k_dia"),
    "taxa_icsap_10k": variacao_pct.get("taxa_icsap_10k_dia"),
}
resumo_etapas["variacao_sazonal_pct"] = resumo_etapas.index.map(variacao_sazonal_map).round(1)

variacao_regional = metricas_por_regiao[metricas_distribuicao].max() - metricas_por_regiao[metricas_distribuicao].min()
resumo_etapas["variacao_geografica_absoluta"] = variacao_regional.round(2)

resumo_etapas.round(2)

,mediana_geral,iqr,pct_outlier,corr_spearman_c_estabelecimento,variacao_sazonal_pct,variacao_geografica_absoluta
taxa_internacao_uti_10k,67.28,40.73,2.1,-0.24,8.1,51.08
taxa_icsap_10k,12.05,11.47,4.4,-0.12,12.1,9.76
pct_icsap,18.32,9.39,2.1,0.02,NaN,6.25
custo_medio_internacao,7933.72,2418.08,1.3,-0.01,NaN,2383.14
dias_medios_uti_internacao,6.35,1.84,2.4,0.21,NaN,1.80


A tabela deixa visual o que já foi dito etapa a etapa: nenhuma métrica tem correlação forte com capacidade instalada (coluna de correlação sempre abaixo de 0,25 em módulo), a variação sazonal é a mais fraca das três dimensões testadas, e a variação geográfica é consistentemente a maior das três — mesmo comparando escalas diferentes (percentual x valor absoluto), a região é o recorte que mais separa município entre si.

### 8.2 O que a tese central consegue e não consegue responder com este dataset

A tese do projeto (município que investe mais em atenção primária gasta menos com internação evitável) depende de cruzar dado de atenção primária (SIA/SIOPS) com o SIH. As duas fontes estão fora do escopo deste dataset (decisão do grupo, complexidade de integração). Por isso, nenhuma etapa desta AED testa a tese diretamente — nenhuma delas tem uma variável de investimento em atenção primária.

O que a AED entrega é evidência indireta, consistente com a tese mas não prova dela: a etapa 7 mostra que a região da capital (que concentra mais estrutura de saúde) tem a menor taxa de ICSAP mediana entre as 11 regiões; a etapa 5 mostra que isso não é simplesmente capacidade hospitalar instalada, já que a correlação entre leito/estabelecimento e as métricas centrais é fraca. É uma pista geográfica compatível com a hipótese, não uma confirmação causal — falta a variável de atenção primária pra fechar o argumento.

### 8.3 Como essas etapas se encaixam no Challenge do grupo

Essa AED é uma frente entre várias do Challenge FIAP-Oracle 2026 do grupo (case "Painel Inteligente de Acesso Hospitalar e Perfil de Atendimento"), que organiza o problema em 4 pilares: Internação, Perfil, Doenças, Sazonalidade por Região.

- **Internação** (volume, tempo, custo): coberto pela etapa 3, as 5 métricas centrais por município x ano.
- **Doenças** (por que internaram, CID-10): coberto pela classificação ICSAP (etapa 1) e pela quebra por grupo CSAP (etapa 6).
- **Sazonalidade por Região** (quando e onde): coberto pelas etapas 6 e 7 juntas — é o achado mais forte desta AED.
- **Perfil** (quem são os pacientes): não coberto por este dataset. A unidade de análise aqui é o bucket agregado (município x diagnóstico x ano x mês), sem campo de paciente individual (idade, sexo, raça/cor). Perfil demográfico de paciente exigiria um recorte com dado não agregado.

O grupo vai discutir, com prós e contras, a possibilidade de incluir perfil de paciente (o SIH bruto tem os campos, ficaram de fora do dataset atual por decisão de agregação) e dado financeiro/SIOPS (fora do escopo atual por complexidade de integração) numa versão futura do projeto.

### Encerramento

Com a etapa 8, a AED do dataset ICSAP-UTI está concluída: 7 dimensões analisadas (perfil, missing, distribuição, outliers, correlação, sazonalidade, geografia), todas com decisão metodológica justificada e achado honesto, inclusive quando o achado foi "não tem correlação forte" ou "não dá pra provar a tese com este dataset". Próximo passo do projeto: passo 9 do guia de métodos estatísticos, fora do escopo desta AED.